In [0]:
SELECT *
FROM workspace.dbo.coffee
LIMIT 3;

--Which product makes the most revenue
SELECT 
    product_detail,
    ROUND(SUM(transaction_qty * unit_price), 2) AS product_Revenue
FROM workspace.dbo.coffee
GROUP BY product_detail
ORDER BY product_Revenue DESC;

/*
What time of day is busiest?Peak hours analysis Column to use: transaction_time + transaction_qty
For this one we need to extract the hour from transaction_time and then label it as:
 Morning / Afternoon / Evening using CASE

*/

SELECT 
    --dayname(transaction_date) AS day_of_The_Week,
    HOUR(transaction_time) AS hour_of_day,
CASE 
    WHEN  hour_of_day  BETWEEN 6 AND 11 THEN 'MORNING'
    WHEN  hour_of_day BETWEEN 12 AND 16 THEN 'AFTERNOON'
    WHEN  hour_of_day BETWEEN 17 AND 20 THEN 'EVENING'
    ELSE 'off_peak'
    END AS time_of_day,
SUM(transaction_qty) as total_items_sold
FROM workspace.dbo.coffee
GROUP BY 
  hour_of_day,
 --dayname(transaction_date),
    CASE
        WHEN HOUR(transaction_time) BETWEEN 6  AND 11 THEN 'Morning'
        WHEN HOUR(transaction_time) BETWEEN 12 AND 16 THEN 'Afternoon'
        WHEN HOUR(transaction_time) BETWEEN 17 AND 20 THEN 'Evening'
        ELSE 'Off Peak'
    END
    ORDER BY total_items_sold DESC;
/*
     So hour 10 (10am) in the Morning period is the busiest! ☕🔥
*/

    --Which store performs best?
  SELECT 
  store_location, 
    ROUND(SUM(transaction_qty* unit_price), 2) as total_revenue_by_store,
    SUM(transaction_qty) as total_items_sold,
    COUNT(DISTINCT transaction_id) AS total_transaction_per_store
  FROM workspace.dbo.coffee
  GROUP BY store_location
  ORDER BY total_revenue_by_store DESC;

/*
Interesting observations:
1. Hell's Kitchen wins on revenue but barely
Only ~R6,000 difference between first and last. These stores are performing almost identically — no store is clearly dominating.
2. Lower Manhattan sells the SAME items as Hell's Kitchen
71,742 vs 71,737 items sold — almost identical volume. But Lower Manhattan makes R6,000 LESS revenue. That means Lower Manhattan is selling cheaper products on average.
3. Lower Manhattan has the least transactions
47,782 vs 50,735 — but still moves the same number of items. So Lower Manhattan customers are buying more items per visit than the other stores.

Boardroom insight: All 3 stores are healthy and consistent. Lower Manhattan customers buy more items per transaction but cheaper products. Hell's Kitchen wins on revenue purely because of slightly higher priced product mix.
*/

/*
How are sales trending over time? Monthly/weekly trends(which month perfroms best)
*/

SELECT 
    MONTHNAME(transaction_date) AS Month_name, 
    MONTH(transaction_date) AS month_number,
    ROUND(SUM(transaction_qty* unit_price), 2) AS total_revenue_per_Month
FROM workspace.dbo.coffee
GROUP BY Month_name, month_number
ORDER BY total_revenue_per_Month DESC;

/*
1. Consistent upward trend — every month is higher than the last without a single dip. The business is growing steadily.
2. Big jump from March to April — revenue jumped ~R20,000 in one month. Something happened there — maybe a new product, promotion, or seasonal demand kicking in.
3. Spring/Summer is peak season — April through June is clearly when the coffee shop makes most of its money. February is the weakest month.
4. June is almost double February — R166k vs R76k. That's a massive seasonal difference for the same business.

Boardroom insight:

"Revenue has grown consistently month over month, nearly doubling from February to June. The business should plan staffing, inventory and promotions around the April–June peak period."
*/

--Question 5: Which category earns the most? (Revenue by category)

SELECT 
    product_category, 
    ROUND(SUM(transaction_qty * unit_price), 2) total_revenue_By_Cat, 
    SUM(transaction_qty) AS Items_sold_in_Cat,
    COUNT(DISTINCT transaction_id) AS num_of_transaction_forThisCat
FROM workspace.dbo.coffee
GROUP BY product_category
ORDER BY total_revenue_By_Cat DESC;

/*
"Coffee and Tea drive the majority of revenue as expected. 
However, Coffee Beans represent a high margin opportunity — low volume but premium pricing.
Flavours is a concern — high volume but extremely low revenue per unit, suggesting a pricing review is needed."

*/

-- What days sell the most? (Day of week patterns)
SELECT 
    dayname(transaction_date) AS day_name,
    dayofweek(transaction_date) AS day_number,
    ROUND(SUM(transaction_qty * unit_price), 2) AS revenue_per_Day,
    COUNT(transaction_id) AS total_transactions,
    SUM(transaction_qty)  AS total_units_sold
FROM workspace.dbo.coffee
GROUP BY day_name, day_number
ORDER BY revenue_per_Day DESC;

/*
"Revenue is remarkably consistent across all 7 days with less than 5% variance. 
The slight weekend dip suggests a predominantly weekday commuter customer base. 
Marketing efforts targeting weekend foot traffic could unlock untapped revenue potential."
*/


--What is the average order value? (Spend per transaction)
SELECT 
COUNT(transaction_id) as total_transactions,
ROUND(SUM(transaction_qty * unit_price),2) AS total_revenue,
ROUND(SUM(transaction_qty * unit_price) / COUNT(transaction_id), 2) AS Average_spend
FROM workspace.dbo.coffee;

/*
That's quite low — which makes sense for a coffee shop. But cross reference with your category data:

Coffee Beans averaged R21 per item
Flavours averaged R0.80 per item

So R4.69 is being dragged down heavily by cheap items like Flavours. 
A customer buying Coffee Beans is spending 4x the average per visit.
*/


--  Which products sell most by volume? (Units sold ranking)
SELECT 
    product_detail,
    SUM(transaction_qty) as items_sold,
    ROUND(SUM(transaction_qty * unit_price), 2) AS product_Revenue,  
    COUNT( DISTINCT transaction_ID) transaction_per_product_detail
FROM workspace.dbo.coffee
GROUP BY product_detail
ORDER BY items_sold DESC
LIMIT 10;

/*
Our highest volume products are not our highest revenue drivers. 
Earl Grey leads in units sold but generates half the revenue per unit of Dark Chocolate Lg.
 The business should consider whether pricing on high volume cheap products like Earl Grey can be optimised without losing customers.
*/



--I want to calssify the spend range, howeverm I frist need to explore my data before I make a decision

--  What is the spend range per transaction?
SELECT 
    MIN(transaction_qty * unit_price) AS min_spend,
    MAX(transaction_qty * unit_price) AS max_spend,
    AVG(transaction_qty * unit_price) AS avg_spend
 FROM workspace.dbo.coffee;

--What is that $360 transaction?
SELECT 
    product_detail,
    product_category,
    transaction_qty,
    unit_price,
    (transaction_qty * unit_price) AS spend
FROM workspace.dbo.coffee
WHERE (transaction_qty * unit_price) = 360
LIMIT 5;

--What are the most expenstive transactioons and how often do they happen
SELECT 
    (transaction_qty * unit_price) AS spend,
    COUNT(transaction_id)  AS num_transactions
FROM workspace.dbo.coffee
GROUP BY spend
ORDER BY spend DESC
LIMIT 20;

--What are the cheapest transactions and how common are they?
SELECT 
    (transaction_qty * unit_price) AS spend,
    COUNT(transaction_id)  AS num_transactions
FROM workspace.dbo.coffee
GROUP BY spend
ORDER BY spend ASC
LIMIT 20;

/*
classify transactions as High or Low based on whether 
they were above or below the average spend of $4.69"
*/
SELECT 
(transaction_qty * unit_price) AS spend,
CASE
    WHEN (transaction_qty * unit_price) >= 20   THEN 'High Spender'
    WHEN (transaction_qty * unit_price) >= 4.69 THEN 'Mid Spender'
    ELSE                                             'Low Spender'
END AS spend_category
FROM workspace.dbo.coffee;

--Classify the days
SELECT 
dayname(transaction_date) AS day_name,
CASE
    WHEN day_name IN('Sun', 'Sat') THEN 'Weekend'
    ELSE   'Weekday'
END AS day_type
FROM workspace.dbo.coffee;



SELECT 

    -- === PRODUCT INFO ===
    product_detail,
    product_category,

    -- === DATE INFO ===
    transaction_date,
    MONTHNAME(transaction_date)             AS month_name,
    MONTH(transaction_date)                 AS month_number,
    DAYNAME(transaction_date)               AS day_name,
    DAYOFWEEK(transaction_date)             AS day_number,

    -- === TIME INFO ===
    HOUR(transaction_time)                  AS hour_of_day,
    CASE 
    WHEN  hour_of_day BETWEEN 6 AND 11 THEN 'MORNING'
    WHEN  hour_of_day BETWEEN 12 AND 16 THEN 'AFTERNOON'
    WHEN  hour_of_day BETWEEN 17 AND 20 THEN 'EVENING'
    ELSE 'off_peak'
    END AS time_of_day,

    CASE
    WHEN day_name IN('Sun', 'Sat') THEN 'Weekend'
    ELSE   'Weekday'
    END AS day_type,
   

    -- === STORE INFO ===
    store_location,

    -- === NUMBERS ===
    COUNT(transaction_id)                   AS total_transactions,
    SUM(transaction_qty)                    AS total_units_sold,
    ROUND(SUM(transaction_qty * unit_price), 2) AS total_revenue,
    ROUND(SUM(transaction_qty * unit_price) / 
          COUNT(transaction_id), 2)         AS avg_order_value,

    -- === PERFORMANCE LABELS ===
    CASE
    WHEN (transaction_qty * unit_price) >= 20   THEN 'High Spender'
    WHEN (transaction_qty * unit_price) >= 4.69 THEN 'Mid Spender'
    ELSE     'Low Spender'
    END AS spend_category

FROM workspace.dbo.coffee
GROUP BY
    product_detail,
    product_category,
    transaction_date,
    MONTHNAME(transaction_date),
    MONTH(transaction_date),
    DAYNAME(transaction_date),
    CASE
    WHEN DAYNAME(transaction_date) IN('Sun', 'Sat') THEN 'Weekend'
    ELSE   'Weekday'
    END,
    DAYOFWEEK(transaction_date),
    HOUR(transaction_time),
    CASE
        WHEN HOUR(transaction_time) BETWEEN 6  AND 11 THEN 'Morning'
        WHEN HOUR(transaction_time) BETWEEN 12 AND 16 THEN 'Afternoon'
        WHEN HOUR(transaction_time) BETWEEN 17 AND 20 THEN 'Evening'
        ELSE 'Off Peak'
    END,
    CASE
    WHEN (transaction_qty * unit_price) >= 20   THEN 'High Spender'
    WHEN (transaction_qty * unit_price) >= 4.69 THEN 'Mid Spender'
    ELSE    'Low Spender'
    END,
    store_location
ORDER BY total_revenue DESC
LIMIT 200000;



